In [6]:
from pathlib import Path

DATA_DIR = Path("/home/ubuntu/data/octopus_v2")

In [15]:
OUTPUT_DIR = Path("./outputs_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
nodes_path = DATA_DIR / "nodes.csv"
edges_path = DATA_DIR / "edges.csv"

In [9]:
from filtering.impl.graph.networkx_graph import NetworkXGraph

In [10]:
graph = NetworkXGraph(
    nodes_path=nodes_path,
    edges_path=edges_path,
    node_id_column_name=None,
    node_ocid_column_name="ocid_node",
    edge_start_column_name="ocid_subject",
    edge_end_column_name="ocid_object",
    edge_time_column_name="timestamp",
    edge_is_time_column_timestamp=True,
    edge_num_occurrences_column_name="number_of_occurrences",
)

In [11]:
from collections import defaultdict

In [13]:
degree_cache = {
    "in_degree": defaultdict(lambda: defaultdict(int)),
    "out_degree": defaultdict(lambda: defaultdict(int)),
}

In [14]:
from tqdm.auto import tqdm

In [18]:
for e in tqdm(graph._g.edges(data=True)):
    timestamp = e[2]["timestamp"]
    year = e[2]["year"]
    number_of_occurrences = e[2]["number_of_occurrences"]

    subj_ocid = graph._g.nodes[e[0]]["ocid_node"]
    obj_ocid = graph._g.nodes[e[1]]["ocid_node"]

    degree_cache["in"][obj_ocid][year] += number_of_occurrences
    degree_cache["out"][subj_ocid][year] += number_of_occurrences

  0%|          | 0/29077032 [00:00<?, ?it/s]

In [19]:
def nested_defaultdicts_to_dicts(nested_defaultdicts):
    if isinstance(nested_defaultdicts, dict):
        return {k: nested_defaultdicts_to_dicts(v) for k, v in nested_defaultdicts.items()}
    return nested_defaultdicts

In [20]:
# Save into pickle format
import pickle

pkl_output_root = OUTPUT_DIR / "pickle"
pkl_output_root.mkdir(parents=True, exist_ok=True)

for direction, cache in degree_cache.items():
    save_path = pkl_output_root / f"{direction}_degree.pkl"
    print(save_path)
    with open(save_path, "wb") as f:
        pickle.dump(nested_defaultdicts_to_dicts(cache), f)

outputs_v2/pickle/train_in_degree.pkl
outputs_v2/pickle/train_out_degree.pkl
outputs_v2/pickle/valid_in_degree.pkl
outputs_v2/pickle/valid_out_degree.pkl
outputs_v2/pickle/test_in_degree.pkl
outputs_v2/pickle/test_out_degree.pkl


In [21]:
# Save into ndjson format
import json

ndjson_output_root = OUTPUT_DIR / "ndjson"
ndjson_output_root.mkdir(parents=True, exist_ok=True)

for direction, cache in degree_cache.items():
    save_path = ndjson_output_root / f"{direction}_degree.ndjson"
    print(save_path)
    with open(save_path, "w") as f:
        for k, v in cache.items():
            json_str = json.dumps({"node_id": k, "degrees": v})
            f.write(json_str)
            f.write("\n")

outputs_v2/ndjson/train_in_degree.ndjson
outputs_v2/ndjson/train_out_degree.ndjson
outputs_v2/ndjson/valid_in_degree.ndjson
outputs_v2/ndjson/valid_out_degree.ndjson
outputs_v2/ndjson/test_in_degree.ndjson
outputs_v2/ndjson/test_out_degree.ndjson
